# 臺北榮民總醫院 SmartCoder 正式 API

[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dechnology/smartcoder-hospital-colab/blob/main/colab/hospitals/tvgh_smartcoder_api.ipynb)

- 院別代號：`tvgh`
- API base URL：`https://fhircsh.itri-nlp.tw/code_api/tvgh`
- 環境：正式服務
- 驗證狀態：正式入口已設定；完整病例整理流程待重新驗證。

執行第一個程式區塊時，只會透過隱藏輸入欄要求該院 API key。Notebook 不會儲存或顯示 key。

In [ ]:
from getpass import getpass
from uuid import uuid4
from datetime import datetime
import json
import time

import requests

HOSPITAL_SLUG = "tvgh"
BASE_URL = "https://fhircsh.itri-nlp.tw/code_api/tvgh"
SERVICE_ENABLED = True
SERVICE_STATUS = "正式入口已設定；完整病例整理流程待重新驗證。"
COLAB_ORIGIN = "https://colab.research.google.com"
EXPECTED_SNOMED_VERSION = "2026-07-01"
EXPECTED_PIPELINE_VERSION = "api-version-0.1.0|txt_ner"
EXPECTED_VOTE_ATTEMPTS = 3
EXPECTED_CONFIDENCE_METHOD = "txt_ner_assertion_filtered_vote_support_min_2"
REQUIRED_RESULT_FIELDS = frozenset((
    "request_id",
    "polished_clinical_note",
    "snomed_codings",
    "processing_metadata",
))
REQUIRED_GET_FIELDS = frozenset((
    "request_id",
    "status",
    "status_code",
    "occurred_at",
    "duration_ms",
    "response",
    "error_detail",
))
REQUIRED_CODING_FIELDS = frozenset((
    "concept_id",
    "term",
    "category",
    "confidence",
    "source",
))
REQUIRED_METADATA_FIELDS = frozenset((
    "polish_model",
    "snomed_version",
    "pipeline_version",
    "vote_attempts",
    "confidence_method",
    "timestamp",
))

CODE_URL = f"{BASE_URL}/api/v1/snomed/coding"
RESULT_URL = f"{BASE_URL}/api/v1/snomed/results/{{request_id}}"

API_KEY = None
if SERVICE_ENABLED:
    API_KEY = getpass(f"請輸入 {HOSPITAL_SLUG} 的 X-API-Key：").strip()
    if not API_KEY:
        raise ValueError("沒有輸入 API key。")

HEADERS = {
    "Content-Type": "application/json",
    "X-API-Key": API_KEY,
    "X-Hospital-Slug": HOSPITAL_SLUG,
    "Origin": COLAB_ORIGIN,
}

print("Hospital:", HOSPITAL_SLUG)
print("BASE_URL:", BASE_URL)
print("Status:", SERVICE_STATUS)
print("API key loaded:", bool(API_KEY))

In [ ]:
def assert_colab_cors(response):
    actual = response.headers["Access-Control-Allow-Origin"]
    assert isinstance(actual, str), "CORS header 型別不符"
    assert actual == COLAB_ORIGIN, (
        f"CORS 不符：預期 {COLAB_ORIGIN}，實際 {actual!r}"
    )


def normalized_text(value):
    assert isinstance(value, str), "病歷內容必須是字串"
    return " ".join(value.split()).casefold()


def assert_iso_timestamp(value, field_name):
    assert isinstance(value, str) and value.strip(), f"{field_name} 必須是非空時間字串"
    try:
        parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError as exc:
        raise AssertionError(f"{field_name} 必須是 ISO 8601 時間") from exc
    assert parsed.tzinfo is not None and parsed.utcoffset() is not None, (
        f"{field_name} 必須包含時區"
    )


def assert_result_contract(payload, expected_request_id, raw_note):
    assert isinstance(payload, dict), "API 回應必須是 JSON object"
    missing_result_fields = REQUIRED_RESULT_FIELDS - set(payload)
    assert not missing_result_fields, (
        f"回應缺少必要欄位：{sorted(missing_result_fields)}"
    )
    assert set(payload) == REQUIRED_RESULT_FIELDS, "回應含有契約外欄位"

    request_id = payload["request_id"]
    assert isinstance(request_id, str), "request_id 必須是字串"
    assert request_id == expected_request_id, "request_id 不一致"

    polished_note = payload["polished_clinical_note"]
    assert isinstance(polished_note, str), "polished_clinical_note 必須是字串"
    assert polished_note.strip(), "整理後病歷不得為空"
    assert normalized_text(polished_note) != normalized_text(raw_note), (
        "整理後病歷不得與原始病歷相同"
    )

    codings = payload["snomed_codings"]
    assert isinstance(codings, list), "snomed_codings 必須是陣列"
    assert codings, "編碼結果不得為空"
    seen_concept_ids = set()
    for index, item in enumerate(codings):
        assert isinstance(item, dict), f"snomed_codings[{index}] 必須是 object"
        missing_coding_fields = REQUIRED_CODING_FIELDS - set(item)
        assert not missing_coding_fields, (
            f"snomed_codings[{index}] 缺少必要欄位："
            f"{sorted(missing_coding_fields)}"
        )
        allowed_coding_fields = REQUIRED_CODING_FIELDS | {"tui"}
        assert set(item) <= allowed_coding_fields, (
            f"snomed_codings[{index}] 含有契約外欄位"
        )

        concept_id = item["concept_id"]
        assert isinstance(concept_id, str) and concept_id.isdigit(), (
            f"snomed_codings[{index}].concept_id 必須是數字字串"
        )
        assert concept_id not in seen_concept_ids, "snomed_codings 不得含重複 concept_id"
        seen_concept_ids.add(concept_id)

        term = item["term"]
        assert isinstance(term, str) and term.strip(), (
            f"snomed_codings[{index}].term 必須是非空字串"
        )
        category = item["category"]
        assert isinstance(category, str) and category in {
            "finding", "disorder", "procedure"
        }, f"snomed_codings[{index}].category 不符合正式編碼契約"
        if "tui" in item and item["tui"] is not None:
            assert isinstance(item["tui"], str) and item["tui"].strip(), (
                f"snomed_codings[{index}].tui 必須是非空字串或 null"
            )
        confidence = item["confidence"]
        assert not isinstance(confidence, bool) and isinstance(confidence, (int, float)), (
            f"snomed_codings[{index}].confidence 必須是數值"
        )
        assert 0.0 < float(confidence) <= 1.0, (
            f"snomed_codings[{index}].confidence 必須大於 0 且不大於 1"
        )
        source = item["source"]
        assert isinstance(source, list), f"snomed_codings[{index}].source 必須是陣列"
        assert source == ["TXT_NER"], "結果含有非 TXT_NER 來源"

    metadata = payload["processing_metadata"]
    assert isinstance(metadata, dict), "processing_metadata 必須是 object"
    assert set(metadata) == REQUIRED_METADATA_FIELDS, (
        "processing_metadata 欄位不符合正式契約"
    )
    polish_model = metadata["polish_model"]
    assert isinstance(polish_model, str) and polish_model.strip(), (
        "polish_model 必須是非空字串"
    )
    assert metadata["snomed_version"] == EXPECTED_SNOMED_VERSION, (
        "snomed_version 不符合鎖定版本"
    )
    assert metadata["pipeline_version"] == EXPECTED_PIPELINE_VERSION, (
        "pipeline_version 不是鎖定的完整 TXT_NER 流程"
    )
    vote_attempts = metadata["vote_attempts"]
    assert not isinstance(vote_attempts, bool) and isinstance(vote_attempts, int), (
        "vote_attempts 必須是整數"
    )
    assert vote_attempts == EXPECTED_VOTE_ATTEMPTS, "NER 必須執行 3 輪"
    assert metadata["confidence_method"] == EXPECTED_CONFIDENCE_METHOD, (
        "confidence_method 不符合三輪投票契約"
    )
    assert_iso_timestamp(metadata["timestamp"], "processing_metadata.timestamp")
    return codings


def public_result(payload):
    assert isinstance(payload, dict), "API 回應必須是 JSON object"
    metadata = payload["processing_metadata"]
    assert isinstance(metadata, dict), "processing_metadata 必須是 object"
    polished_note = payload["polished_clinical_note"]
    assert isinstance(polished_note, str), "polished_clinical_note 必須是字串"
    codings = payload["snomed_codings"]
    assert isinstance(codings, list), "snomed_codings 必須是陣列"
    public_metadata = {
        key: metadata[key]
        for key in (
            "snomed_version",
            "pipeline_version",
            "vote_attempts",
            "confidence_method",
        )
    }
    return {
        "request_id": payload["request_id"],
        "polished_clinical_note": polished_note,
        "snomed_codings": codings,
        "processing_metadata": public_metadata,
    }


def run_full_flow_case():
    request_id = str(uuid4())
    payload = {
        "request_id": request_id,
        "raw_clinical_note": "患者胸痛持續兩週，否認咳嗽。",
        "output_format": "simple",
    }

    started_at = time.perf_counter()
    post_response = requests.post(
        CODE_URL,
        headers=HEADERS,
        json=payload,
        timeout=240,
        allow_redirects=False,
    )
    elapsed_seconds = time.perf_counter() - started_at
    print(
        f"POST HTTP {post_response.status_code}｜完整流程 "
        f"{elapsed_seconds:.2f} 秒"
    )
    post_response.raise_for_status()
    assert post_response.status_code == 200, "POST 必須直接回傳 HTTP 200"
    assert post_response.url == CODE_URL, "POST 不得轉向其他網址"
    assert_colab_cors(post_response)

    post_payload = post_response.json()
    codings = assert_result_contract(
        post_payload,
        expected_request_id=request_id,
        raw_note=payload["raw_clinical_note"],
    )
    assert any(
        item["concept_id"] == "29857009" for item in codings
    ), "未辨識出胸痛 SNOMED 概念"

    get_response = requests.get(
        RESULT_URL.format(request_id=request_id),
        headers=HEADERS,
        timeout=60,
        allow_redirects=False,
    )
    print("GET HTTP", get_response.status_code)
    get_response.raise_for_status()
    assert get_response.status_code == 200, "GET 必須直接回傳 HTTP 200"
    assert get_response.url == RESULT_URL.format(request_id=request_id), (
        "GET 不得轉向其他網址"
    )
    assert_colab_cors(get_response)

    get_payload = get_response.json()
    assert isinstance(get_payload, dict), "GET 回應必須是 JSON object"
    assert set(get_payload) == REQUIRED_GET_FIELDS, "GET 回應欄位不符合正式契約"
    get_request_id = get_payload["request_id"]
    assert isinstance(get_request_id, str), "GET request_id 必須是字串"
    assert get_request_id == request_id, "GET request_id 不一致"
    get_status = get_payload["status"]
    assert isinstance(get_status, str), "GET status 必須是字串"
    assert get_status == "completed", "GET 任務狀態不是 completed"
    get_status_code = get_payload["status_code"]
    assert not isinstance(get_status_code, bool) and isinstance(get_status_code, int), (
        "GET status_code 必須是整數"
    )
    assert get_status_code == 200, "GET 儲存的任務狀態碼不是 200"
    duration_ms = get_payload["duration_ms"]
    assert not isinstance(duration_ms, bool) and isinstance(duration_ms, int), (
        "GET duration_ms 必須是整數"
    )
    assert duration_ms >= 0, "GET duration_ms 不得為負數"
    occurred_at = get_payload["occurred_at"]
    assert isinstance(occurred_at, str) and occurred_at.strip(), (
        "GET occurred_at 必須是非空時間字串"
    )
    assert_iso_timestamp(occurred_at, "GET occurred_at")
    assert get_payload["error_detail"] is None, "GET 完成結果不得含錯誤訊息"
    get_result = get_payload["response"]
    assert_result_contract(
        get_result,
        expected_request_id=request_id,
        raw_note=payload["raw_clinical_note"],
    )
    assert get_result == post_payload, "GET 與 POST 完整結果不一致"

    print(json.dumps(public_result(post_payload), ensure_ascii=False, indent=2))

In [ ]:
if not SERVICE_ENABLED:
    raise RuntimeError(SERVICE_STATUS)

run_full_flow_case()
print("正式流程驗收通過")